In [51]:
import requests
from datetime import date

records = []

start_year = 2025
start_month = 1

end_year = 2025
end_month = 3

year = start_year
month = start_month

while (year, month) <= (end_year, end_month):

    # First day of the month
    start_date = date(year, month, 1)

    # First day of the next month
    if month == 12:
        next_year = year + 1
        next_month = 1
    else:
        next_year = year
        next_month = month + 1

    end_date = date(next_year, next_month, 1)

    print("Downloading:", start_date, "to", end_date)

    url = (
        "https://earthquake.usgs.gov/fdsnws/event/1/query"
        f"?format=geojson"
        f"&starttime={start_date}"
        f"&endtime={end_date}"
    )

    response = requests.get(url)

    data = response.json()

    print("Earthquakes:", len(data["features"]))

    # Move to next month
    year = next_year
    month = next_month

print("Total records:", len(records))

Downloading: 2025-01-01 to 2025-02-01
Earthquakes: 11632
Downloading: 2025-02-01 to 2025-03-01
Earthquakes: 11505
Downloading: 2025-03-01 to 2025-04-01
Earthquakes: 11861
Total records: 0


In [45]:
from datetime import datetime
from dateutil.relativedelta import relativedelta

end_date = datetime.now() # today date
start_date = end_date - relativedelta(months=5) # 5 years/months ago

current_date = start_date # 5 months ago date

while current_date < end_date: # yes definitely less than today date, so we can loop through the months

    next_date = current_date + relativedelta(months=1) # increment the current date by 1 month
    print(current_date.date(), "→", next_date.date())
    
    if next_date > end_date: # if the next date is greater than today date, then we set the next date to today date
        next_date = end_date  

    current_date = next_date #after the loop, we set the current date to the next date, so that we can loop through the months again

2026-05-06 → 2026-06-06
2026-06-06 → 2026-07-06
2026-07-06 → 2026-08-06
2026-08-06 → 2026-09-06
2026-09-06 → 2026-10-06


In [ ]:
import requests
from datetime import datetime
from dateutil.relativedelta import relativedelta

url = "https://earthquake.usgs.gov/fdsnws/event/1/query"

end_date = datetime.now()
start_date = end_date - relativedelta(months=2)
records = []

current_date = start_date

while current_date < end_date:

    next_date = current_date + relativedelta(months=1)

    if next_date > end_date:
        next_date = end_date 

    parameters = {
        "format": "geojson",
        "starttime": current_date.strftime("%Y-%m-%dT%H:%M:%S"),
        "endtime": next_date.strftime("%Y-%m-%dT%H:%M:%S"),
        "limit": 20000,
        "offset": 1,
        "orderby": "time-asc"
    }
 
    response = requests.get(url,params=parameters,timeout=60)
    current_date = next_date    
    data = response.json()
    #print(f"DATA of {current_date.date()} to {next_date.date()}: {data}")
    records.append(data)
    print(records)

In [48]:
records = [item for sublist in records for item in sublist]

In [49]:
records

['type',
 'metadata',
 'features',
 'bbox',
 'type',
 'metadata',
 'features',
 'bbox']

In [20]:
import requests
import pandas as pd
from datetime import datetime
from dateutil.relativedelta import relativedelta
# 1. USGS API
url = "https://earthquake.usgs.gov/fdsnws/event/1/query"
# 2. 5-year date range
end_date = datetime.now()
start_date = end_date - relativedelta(months=3)
current_date = start_date
# 6. Download month by month
while current_date < end_date:
    next_date = current_date + relativedelta(months=1)
    if next_date > end_date:
        next_date = end_date  
    #print("Downloading:", current_date.date(), "to", next_date.date())
    offset = 1
    # 7. Handle pagination
    while True:
        parameters = {
            "format": "geojson",
            "starttime": current_date.strftime("%Y-%m-%dT%H:%M:%S"),
            "endtime": next_date.strftime("%Y-%m-%dT%H:%M:%S"),
            "limit": 20000,
            "offset": offset,
            "orderby": "time-asc"
        }
        response = requests.get(
            url,
            params=parameters,
            timeout=60
        )
        #print("Status:", response.status_code)
#        response.raise_for_status()
        data = response.json()
        features = data["features"]
 #       print("Received:", len(features))
        # 8. Stop if no data
        if len(features) == 0:
            break
        # 11. Pagination
        if len(features) < 20000:
            break
        offset += 20000
    # 12. Move to next month
    current_date = next_date
# 13. Finished
print("3 months DOWNLOAD COMPLETE")

3 months DOWNLOAD COMPLETE


In [36]:
import requests
import pandas as pd
import pymysql

from datetime import datetime
from dateutil.relativedelta import relativedelta


# --------------------------------
# 1. USGS API
# --------------------------------

url = "https://earthquake.usgs.gov/fdsnws/event/1/query"


# --------------------------------
# 2. 5-year date range
# --------------------------------

end_date = datetime.now()
start_date = end_date - relativedelta(months=5)

current_date = start_date


# --------------------------------
# 6. Download month by month
# --------------------------------
monthlyrecords = []
while current_date < end_date:

    next_date = current_date + relativedelta(months=1)

    if next_date > end_date:
        next_date = end_date

    print()
    print("--------------------------------")
    print("Downloading:", current_date.date(), "to", next_date.date())
    print("--------------------------------")


    offset = 1


    # --------------------------------
    # 7. Handle pagination
    # --------------------------------
    actualrecords = []
    while True:
        

        parameters = {
            "format": "geojson",
            "starttime": current_date.strftime("%Y-%m-%dT%H:%M:%S"),
            "endtime": next_date.strftime("%Y-%m-%dT%H:%M:%S"),
            "limit": 20000,
            "offset": offset,
            "orderby": "time-asc"
        }


        response = requests.get(
            url,
            params=parameters,
            timeout=60
        )
        print("Status:", response.status_code)
        #response.raise_for_status()

        data = response.json()
        monthlyrecords.append(data)
        actualrecords.append(data) 


        features = data["features"]

        print("Received:", len(features))
        


        # --------------------------------
        # 8. Stop if no data
        # --------------------------------

        if len(features) == 0:
            break
# --------------------------------
        # 11. Pagination
        # --------------------------------

        if len(features) < 20000:
            break

        offset += 20000


    # --------------------------------
    # 12. Move to next month
    # --------------------------------

    current_date = next_date


# --------------------------------
# 13. Finished
# --------------------------------

print()
print("====================================")
print("5-months DOWNLOAD COMPLETE")


--------------------------------
Downloading: 2026-05-06 to 2026-06-06
--------------------------------
Status: 200
Received: 11532

--------------------------------
Downloading: 2026-06-06 to 2026-07-06
--------------------------------
Status: 200
Received: 11979

--------------------------------
Downloading: 2026-07-06 to 2026-08-06
--------------------------------
Status: 200
Received: 11928

--------------------------------
Downloading: 2026-08-06 to 2026-09-06
--------------------------------
Status: 200
Received: 12401

--------------------------------
Downloading: 2026-09-06 to 2026-10-06
--------------------------------
Status: 200
Received: 10438

5-months DOWNLOAD COMPLETE


In [43]:
type(actualrecords)

list

In [44]:
len(actualrecords)

1

In [42]:
monthlyrecords[0]

{'type': 'FeatureCollection',
 'metadata': {'generated': 1791283519000,
  'url': 'https://earthquake.usgs.gov/fdsnws/event/1/query?format=geojson&starttime=2026-05-06T16%3A15%3A16&endtime=2026-06-06T16%3A15%3A16&limit=20000&offset=1&orderby=time-asc',
  'title': 'USGS Earthquakes',
  'status': 200,
  'api': '2.7.0',
  'limit': 20000,
  'offset': 1},
 'features': [{'type': 'Feature',
   'properties': {'mag': 1.96,
    'place': '1 km N of Lluveras, Puerto Rico',
    'time': 1778084118690,
    'updated': 1778086156990,
    'tz': None,
    'url': 'https://earthquake.usgs.gov/earthquakes/eventpage/pr71515953',
    'detail': 'https://earthquake.usgs.gov/fdsnws/event/1/query?eventid=pr71515953&format=geojson',
    'felt': None,
    'cdi': None,
    'mmi': None,
    'alert': None,
    'status': 'reviewed',
    'tsunami': 0,
    'sig': 59,
    'net': 'pr',
    'code': '71515953',
    'ids': ',pr71515953,',
    'sources': ',pr,',
    'types': ',origin,phase-data,',
    'nst': 5,
    'dmin': 0.07

In [ ]:
#data
#type(data)
data.keys()

dict_keys(['type', 'metadata', 'features', 'bbox'])

In [5]:
data['type']

'FeatureCollection'

In [6]:
data['metadata']

{'generated': 1791273785000,
 'url': 'https://earthquake.usgs.gov/fdsnws/event/1/query?format=geojson&starttime=2026-09-06T13%3A28%3A19&endtime=2026-10-06T13%3A28%3A19&limit=20000&offset=1&orderby=time-asc',
 'title': 'USGS Earthquakes',
 'status': 200,
 'api': '2.7.0',
 'limit': 20000,
 'offset': 1}

In [7]:
data['bbox']

[-179.9919, -63.5619, -3.43, 179.9549, 82.6385, 665.057]

In [11]:
data['features']

[{'type': 'Feature',
  'properties': {'mag': 1.5,
   'place': '41 km ENE of Pedro Bay, Alaska',
   'time': 1788701360531,
   'updated': 1788985034682,
   'tz': None,
   'url': 'https://earthquake.usgs.gov/earthquakes/eventpage/aka2026rrcksw',
   'detail': 'https://earthquake.usgs.gov/fdsnws/event/1/query?eventid=aka2026rrcksw&format=geojson',
   'felt': None,
   'cdi': None,
   'mmi': None,
   'alert': None,
   'status': 'reviewed',
   'tsunami': 0,
   'sig': 35,
   'net': 'ak',
   'code': 'a2026rrcksw',
   'ids': ',aka2026rrcksw,',
   'sources': ',ak,',
   'types': ',origin,phase-data,',
   'nst': 22,
   'dmin': 0.2,
   'rms': 0.8,
   'gap': 78,
   'magType': 'ml',
   'type': 'earthquake',
   'title': 'M 1.5 - 41 km ENE of Pedro Bay, Alaska'},
  'geometry': {'type': 'Point', 'coordinates': [-153.427, 59.927, 123]},
  'id': 'aka2026rrcksw'},
 {'type': 'Feature',
  'properties': {'mag': 0.93,
   'place': '11 km WNW of Cobb, CA',
   'time': 1788701482240,
   'updated': 1788703041244,
   

In [12]:
type(data['features'])

list

In [13]:
len(data['features'])

10446

In [15]:
data["features"][10445]

{'type': 'Feature',
 'properties': {'mag': 0.9,
  'place': '7 km NNE of Mecca, CA',
  'time': 1791273370640,
  'updated': 1791273577697,
  'tz': None,
  'url': 'https://earthquake.usgs.gov/earthquakes/eventpage/ci41343959',
  'detail': 'https://earthquake.usgs.gov/fdsnws/event/1/query?eventid=ci41343959&format=geojson',
  'felt': None,
  'cdi': None,
  'mmi': None,
  'alert': None,
  'status': 'automatic',
  'tsunami': 0,
  'sig': 12,
  'net': 'ci',
  'code': '41343959',
  'ids': ',ci41343959,',
  'sources': ',ci,',
  'types': ',nearby-cities,origin,phase-data,',
  'nst': 7,
  'dmin': 0.479,
  'rms': 0.31,
  'gap': 317,
  'magType': 'ml',
  'type': 'earthquake',
  'title': 'M 0.9 - 7 km NNE of Mecca, CA'},
 'geometry': {'type': 'Point',
  'coordinates': [-116.062166666667, 33.633, -1.01]},
 'id': 'ci41343959'}

In [16]:
type(data['features'][10445])

dict

In [17]:
data['features'][10445].keys()

dict_keys(['type', 'properties', 'geometry', 'id'])

In [18]:
data['features'][10445]['geometry']

{'type': 'Point', 'coordinates': [-116.062166666667, 33.633, -1.01]}

In [19]:
data["features"][10445]['properties']

{'mag': 0.9,
 'place': '7 km NNE of Mecca, CA',
 'time': 1791273370640,
 'updated': 1791273577697,
 'tz': None,
 'url': 'https://earthquake.usgs.gov/earthquakes/eventpage/ci41343959',
 'detail': 'https://earthquake.usgs.gov/fdsnws/event/1/query?eventid=ci41343959&format=geojson',
 'felt': None,
 'cdi': None,
 'mmi': None,
 'alert': None,
 'status': 'automatic',
 'tsunami': 0,
 'sig': 12,
 'net': 'ci',
 'code': '41343959',
 'ids': ',ci41343959,',
 'sources': ',ci,',
 'types': ',nearby-cities,origin,phase-data,',
 'nst': 7,
 'dmin': 0.479,
 'rms': 0.31,
 'gap': 317,
 'magType': 'ml',
 'type': 'earthquake',
 'title': 'M 0.9 - 7 km NNE of Mecca, CA'}